# NGLab Tutorial #10: Advanced Topics

Explore cutting-edge techniques: custom reward shaping, model ensembles, and uncertainty quantification.

## Learning Objectives

1. Implement custom reward functions
2. Build model ensembles
3. Quantify prediction uncertainty
4. Advanced debugging and profiling

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import deque

np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch: {torch.__version__}")
print(f"Ready for advanced techniques!")

## 1. Custom Reward Shaping

Design reward functions that optimize for specific objectives:

### Sharpe Ratio Reward

$$
R_{\text{Sharpe}} = \frac{\mathbb{E}[r]}{\sigma(r) + \epsilon}
$$

In [ ]:
class SharpeReward:
    """Rolling Sharpe ratio as reward."""
    
    def __init__(self, window=100):
        self.returns_buffer = deque(maxlen=window)
    
    def compute(self, portfolio_value, prev_value):
        # Calculate return
        ret = (portfolio_value - prev_value) / (prev_value + 1e-8)
        self.returns_buffer.append(ret)
        
        if len(self.returns_buffer) < 10:
            return ret  # Not enough data
        
        # Sharpe = mean / std
        mean_ret = np.mean(self.returns_buffer)
        std_ret = np.std(self.returns_buffer) + 1e-8
        
        return mean_ret / std_ret

class SortinoReward:
    """Sortino ratio: penalizes downside volatility only."""
    
    def __init__(self, window=100, target_return=0.0):
        self.returns_buffer = deque(maxlen=window)
        self.target_return = target_return
    
    def compute(self, portfolio_value, prev_value):
        ret = (portfolio_value - prev_value) / (prev_value + 1e-8)
        self.returns_buffer.append(ret)
        
        if len(self.returns_buffer) < 10:
            return ret
        
        returns_array = np.array(self.returns_buffer)
        downside_returns = returns_array[returns_array < self.target_return]
        
        if len(downside_returns) == 0:
            return np.mean(returns_array)  # No downside
        
        downside_deviation = np.std(downside_returns) + 1e-8
        return np.mean(returns_array) / downside_deviation

# Test
sharpe_reward = SharpeReward()
sortino_reward = SortinoReward()

# Simulate returns
for i in range(100):
    prev_val = 100000
    curr_val = prev_val * (1 + np.random.randn() * 0.01)
    
    sharpe_r = sharpe_reward.compute(curr_val, prev_val)
    sortino_r = sortino_reward.compute(curr_val, prev_val)

print(f"\nFinal Sharpe Reward: {sharpe_r:.4f}")
print(f"Final Sortino Reward: {sortino_r:.4f}")

## 2. Model Ensembles

Combine multiple models for robust predictions:

In [ ]:
class SimplePredictor(nn.Module):
    """Base predictor model."""
    def __init__(self, input_dim=60, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, x):
        return self.net(x)

class EnsembleModel:
    """Ensemble of multiple models."""
    
    def __init__(self, models, aggregation='mean', weights=None):
        self.models = models
        self.aggregation = aggregation
        self.weights = weights or [1.0 / len(models)] * len(models)
    
    def predict(self, x):
        """Aggregate predictions."""
        predictions = []
        
        for model in self.models:
            model.eval()
            with torch.no_grad():
                pred = model(x)
                predictions.append(pred)
        
        predictions = torch.stack(predictions)
        
        if self.aggregation == 'mean':
            return torch.mean(predictions, dim=0)
        elif self.aggregation == 'weighted':
            weights_tensor = torch.tensor(self.weights).view(-1, 1, 1)
            return torch.sum(predictions * weights_tensor, dim=0)
        elif self.aggregation == 'median':
            return torch.median(predictions, dim=0).values
        else:
            raise ValueError(f"Unknown aggregation: {self.aggregation}")
    
    def predict_with_uncertainty(self, x):
        """Return mean prediction + std (uncertainty)."""
        predictions = []
        
        for model in self.models:
            model.eval()
            with torch.no_grad():
                pred = model(x)
                predictions.append(pred)
        
        predictions = torch.stack(predictions)
        mean = torch.mean(predictions, dim=0)
        std = torch.std(predictions, dim=0)
        
        return mean, std

# Create ensemble
models = [SimplePredictor(hidden_dim=64) for _ in range(5)]
ensemble = EnsembleModel(models, aggregation='weighted', weights=[0.3, 0.25, 0.2, 0.15, 0.1])

# Test prediction
test_input = torch.randn(10, 60)
pred_mean, pred_std = ensemble.predict_with_uncertainty(test_input)

print(f"\nEnsemble created with {len(models)} models")
print(f"Prediction shape: {pred_mean.shape}")
print(f"Mean uncertainty(std): {pred_std.mean().item():.4f}")

## 3. Uncertainty Quantification

Visualize prediction intervals:

In [ ]:
# Generate test sequence
true_values = np.sin(np.linspace(0, 4*np.pi, 100)) + np.random.randn(100) * 0.1
test_data = torch.FloatTensor(true_values[:60]).unsqueeze(0)

# Get ensemble predictions with uncertainty
predictions = []
uncertainties = []

for i in range(40):  # Forecast 40 steps
    pred_mean, pred_std = ensemble.predict_with_uncertainty(test_data)
    predictions.append(pred_mean.item())
    uncertainties.append(pred_std.item())
    
    # Roll window (simplified: just use prediction)
    test_data = torch.cat([test_data[:, 1:], pred_mean.unsqueeze(0)], dim=1)

predictions = np.array(predictions)
uncertainties = np.array(uncertainties)

# Plot with confidence intervals
plt.figure(figsize=(14, 6))

x = np.arange(len(predictions))
plt.plot(x, predictions, 'b-', linewidth=2, label='Ensemble Prediction')
plt.fill_between(x, 
                 predictions - 2*uncertainties, 
                 predictions + 2*uncertainties,
                 alpha=0.3, color='blue', label='95% Confidence')
plt.plot(x, true_values[60:60+len(predictions)], 'r--', linewidth=2, label='True Values', alpha=0.7)

plt.title('Ensemble Forecasting with Uncertainty', fontsize=14, fontweight='bold')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Advanced Debugging Tips

### Gradient Flow Visualization

In [ ]:
def plot_grad_flow(named_parameters):
    """Visualize gradient flow through layers."""
    ave_grads = []
    max_grads = []
    layers = []
    
    for n, p in named_parameters:
        if (p.requires_grad) and ("bias" not in n) and p.grad is not None:
            layers.append(n)
            ave_grads.append(p.grad.abs().mean().cpu())
            max_grads.append(p.grad.abs().max().cpu())
    
    plt.figure(figsize=(12, 6))
    plt.bar(np.arange(len(max_grads)), max_grads, alpha=0.5, lw=1, color="c")
    plt.bar(np.arange(len(max_grads)), ave_grads, alpha=0.5, lw=1, color="b")
    plt.hlines(0, 0, len(ave_grads)+1, lw=2, color="k")
    plt.xticks(range(0, len(ave_grads), 1), layers, rotation="vertical")
    plt.xlim(left=0, right=len(ave_grads))
    plt.ylim(bottom=-0.001, top=0.02)
    plt.xlabel("Layers")
    plt.ylabel("Gradient")
    plt.title("Gradient Flow")
    plt.grid(True, alpha=0.3)
    plt.legend(["max-gradient", "mean-gradient"])
    plt.tight_layout()
    plt.show()

# Example: compute gradients
model = SimplePredictor()
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()

# Dummy forward/backward
x = torch.randn(32, 60)
y = torch.randn(32, 1)

optimizer.zero_grad()
pred = model(x)
loss = criterion(pred, y)
loss.backward()

# Visualize
plot_grad_flow(model.named_parameters())

## 5. Key Takeaways

### Best Practices

1. **Custom Rewards**: Design rewards aligned with your business objective (Sharpe, Sortino, etc.)
2. **Ensembles**: Use 5-10 models for robust predictions
3. **Uncertainty**: Always quantify prediction confidence
4. **Debugging**: Monitor gradients, activations, and loss landscapes

### Production Checklist

- ✅ Backtested on 2+ years of data
- ✅ Slippage and transaction costs included
- ✅ Risk limits (VaR, drawdown) enforced
- ✅ Model uncertainty quantified
- ✅ Real-time monitoring (Prometheus/Grafana)
- ✅ Fail-safes for extreme market conditions

## Summary

In this notebook, you learned:

✅ Custom reward functions (Sharpe, Sortino)  
✅ Model ensembling strategies  
✅ Uncertainty quantification  
✅ Advanced debugging techniques  

## 🎉 Congratulations!

You've completed all 10 NGLab tutorial notebooks! You now have the skills to:
- Build and train deep RL trading agents
- Optimize hyperparameters with DEHB
- Deploy multi-agent simulations
- Backtest strategies with realistic costs
- Apply advanced techniques for production systems

### Next Steps

- Check out [TUTORIAL.md](../TUTORIAL.md) for deeper technical dives
- Explore [ARCHITECTURE.md](../ARCHITECTURE.md) for system design
- Review [AGENTS.md](../AGENTS.md) for agent algorithms

**Happy Trading! 🚀**

---